In [24]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import trange, tqdm
from sklearn.metrics import pairwise_distances

import matplotlib.pyplot as plt
import seaborn as sns

# Pasive Data

In [25]:
directory_name = 'MEC_04_005 (OPTIR Pilot Passive 2024 Data)/Raw Data/' 
ir_export_filename =  "no_interpolate_equiv_ir.csv"  ###enter your desired export Raman csv filename, in quotes
raman_export_filename =  "no_interpolate_equiv_raman.csv"  ###enter your desired export Raman csv filename, in quotes

path = 'MEC_04_005 (OPTIR Pilot Passive 2024 Data)/Raw Data/'  #'MEC_04_005 (OPTIR Pilot Passive 2024 Data)/Raw Data/'
folders = os.listdir(path)
files_list = []

for folder in folders:
    if folder[-4:] in ["xlsx", "tore"]:
        continue
    dirs = os.listdir(path+folder)
    for d in dirs:
        if d[-4:] in ["tore"]:
            continue
        files = os.listdir(path+folder+'/'+d)
        files_list = files_list+[folder+'/'+d+'/'+f for f in files if f[-3:] == 'csv']

files_list

['08 August 2024/FHS_20240911/FHS_passive_August2024_B4_FOV4.csv',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_FOV5.csv',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_FOV7.csv',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_FOV6.csv',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_FOV2.csv',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_FOV3.csv',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_FOV1.csv',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_FOV8.csv',
 '08 August 2024/FHS_20240911/FHS_passive_August2024_B4_FOV9.csv',
 '08 August 2024/IAO_20240917/IAO_passive_August2024_K11_FOV5.csv',
 '08 August 2024/IAO_20240917/IAO_passive_August2024_K11_FOV4.csv',
 '08 August 2024/IAO_20240917/IAO_passive_August2024_K11_FOV6.csv',
 '08 August 2024/IAO_20240917/IAO_passive_August2024_K11_FOV3.csv',
 '08 August 2024/IAO_20240917/IAO_passive_August2024_K11_FOV2.csv',
 '08 August 2024/IAO_20240917/IAO_passive_August2024_K11_

In [26]:
df_list = []
prev_mosaic = ''
count = 1
for f in files_list:
    df = pd.read_csv(path + f)
    df.columns = ['feat_id', 'map_x', 'map_y', 'size', 'area', 'perimeter', 'circularity']
    #mosaic = files_codes.loc[f, 'mosaic']
    mosaic = '/'.join(f.split('/')[:-1])
    #mosaic = f[:-4]
    if mosaic == prev_mosaic:
        count = count + 1
    df['id'] = [f[:-4] + '_' +
                str(i + 1) for i in range(df.shape[0])]
    
    #df['id'] = [files_codes.loc[f, 'brand'] + '_' + 
    #            files_codes.loc[f, 'material'] + '_' + 
    #            files_codes.loc[f, 'run'] + '_' + 
    #            files_codes.loc[f, 'mosaic'] + '_' +
    #            str(count) + '_' +
    #            str(i + 1) for i in range(df.shape[0])]
    
    df['id_file'] = [mosaic for i in range(df.shape[0])]
    
    #df['brand'] = files_codes.loc[f, 'brand']
    #df['run'] = files_codes.loc[f, 'run']
    #df['mosaic'] = files_codes.loc[f, 'mosaic']
    df_list.append(df.iloc[:,:])
    #prev_mosaic = mosaic
    
coord_df = pd.concat(df_list,axis=0).reset_index(drop=True)
coord_df

,feat_id,map_x,map_y,size,area,perimeter,circularity,id,id_file
0,1,15675.77,1516.8670,4.327407,14.628250,17.526080,0.598301,08 August 2024/FHS_20240911/FHS_passive_August...,08 August 2024/FHS_20240911
1,2,15518.41,1531.3790,6.125319,29.510470,21.815600,0.779002,08 August 2024/FHS_20240911/FHS_passive_August...,08 August 2024/FHS_20240911
2,3,15650.36,1540.5450,2.041465,3.301516,7.062470,0.831565,08 August 2024/FHS_20240911/FHS_passive_August...,08 August 2024/FHS_20240911
3,4,15659.36,1539.4030,8.672388,46.475180,40.858250,0.349750,08 August 2024/FHS_20240911/FHS_passive_August...,08 August 2024/FHS_20240911
4,5,15652.15,1546.3150,1.203046,1.108971,4.841497,0.594370,08 August 2024/FHS_20240911/FHS_passive_August...,08 August 2024/FHS_20240911
...,...,...,...,...,...,...,...,...,...
10622,42,21437.92,-584.0508,1.888275,2.818986,6.573632,0.819556,06 June 2024/IAO_20240725/IAO_passive_June2024...,06 June 2024/IAO_20240725
10623,43,21376.93,-576.4603,0.752801,0.490995,2.880727,0.743308,06 June 2024/IAO_20240725/IAO_passive_June2024...,06 June 2024/IAO_20240725
10624,44,21377.12,-575.0192,9.650741,72.912700,35.622620,0.721852,06 June 2024/IAO_20240725/IAO_passive_June2024...,06 June 2024/IAO_20240725
10625,45,21404.03,-568.6928,1.367004,1.472984,4.657506,0.853076,06 June 2024/IAO_20240725/IAO_passive_June2024...,06 June 2024/IAO_20240725


In [27]:
preposs_data_raman = pd.read_csv('passive_data_raman_specy.csv')
del preposs_data_raman['Unnamed: 0']
preposs_data_raman

,600.0,602.0,604.0,606.0,608.0,610.0,612.0,614.0,616.0,618.0,...,3492.0,3494.0,3496.0,3498.0,map_x,map_y,id,wn_saturated,no_signal,id_file
0,-0.199424,-0.165922,-0.124521,-0.082595,-0.041191,-0.000810,0.032492,0.061125,0.068892,0.067682,...,-0.199424,-0.199424,-0.199424,-0.199424,15435.200195,1589.199951,FHS_passive_August2024_B4_20240910_1,0,0,08 August 2024/FHS_20240911
1,-0.159909,-0.145882,-0.130207,-0.096940,-0.062625,-0.026258,0.002609,0.026532,0.029744,0.026403,...,-0.180029,-0.180029,-0.180029,-0.180029,15436.750000,1599.699951,FHS_passive_August2024_B4_20240910_2,0,0,08 August 2024/FHS_20240911
2,-0.145047,-0.137766,-0.131138,-0.095330,-0.060413,-0.027241,-0.002443,0.016833,0.018447,0.016011,...,-0.176024,-0.176024,-0.176024,-0.176024,15392.150391,1603.900024,FHS_passive_August2024_B4_20240910_3,0,0,08 August 2024/FHS_20240911
3,-0.175573,-0.161105,-0.142401,-0.113194,-0.081894,-0.046502,-0.018065,0.005786,0.001212,-0.008950,...,-0.175573,-0.175573,-0.175573,-0.175573,15437.599609,1606.650024,FHS_passive_August2024_B4_20240910_4,0,0,08 August 2024/FHS_20240911
4,-0.191092,-0.175071,-0.153104,-0.103835,-0.063849,-0.042011,-0.020313,0.001293,0.012876,0.017845,...,-0.191092,-0.191092,-0.191092,-0.191092,15439.450195,1611.050049,FHS_passive_August2024_B4_20240910_5,0,0,08 August 2024/FHS_20240911
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8181,-0.164180,-0.152205,-0.134102,-0.100990,-0.066696,-0.031450,-0.006181,0.015009,0.015447,0.011999,...,-0.179955,-0.179955,-0.179955,-0.179955,7783.100098,-485.600006,IAO_passive_June2024_G7_20240801_39,0,0,06 June 2024/IAO_20240725
8182,-0.193101,-0.148955,-0.109848,-0.083081,-0.056534,-0.030163,-0.009836,0.007955,0.012440,0.011380,...,-0.194242,-0.194242,-0.194242,-0.194242,7778.299805,-487.200012,IAO_passive_June2024_G7_20240801_40,0,0,06 June 2024/IAO_20240725
8183,-0.172771,-0.161864,-0.145300,-0.114880,-0.079252,-0.039431,-0.013829,0.006928,0.008910,0.009713,...,-0.173053,-0.173053,-0.173053,-0.173053,7740.350098,-478.950012,IAO_passive_June2024_G7_20240801_41,0,0,06 June 2024/IAO_20240725
8184,-0.174525,-0.148922,-0.122501,-0.094080,-0.068712,-0.045801,-0.023883,-0.002829,0.011055,0.014692,...,-0.175187,-0.175187,-0.175187,-0.175187,7828.049805,-470.100006,IAO_passive_June2024_G7_20240801_42,0,0,06 June 2024/IAO_20240725


In [31]:
spectra_matches_list = []
for id_ in np.unique(preposs_data_raman['id_file']):
    
    if coord_df.query('id_file == @id_').shape[0] == 0:
        continue
    id_matches = np.argmin(pairwise_distances(preposs_data_raman.query('id_file == @id_').loc[:,['map_x', 'map_y']], 
                                              coord_df.query('id_file == @id_').loc[:,['map_x', 'map_y']]),1)


    coord_matches = coord_df.query('id_file == @id_').iloc[id_matches, :].reset_index(drop=True)
    spectra_matches = (preposs_data_raman.
                       query('id_file == @id_').
                       reset_index(drop=True).
                       merge(coord_matches, left_index=True, right_index=True).iloc[:, -15:])
    
    spectra_matches['distance'] = np.sqrt((spectra_matches['map_x_x'] - spectra_matches['map_x_y']) ** 2 + 
                                          (spectra_matches['map_y_x'] - spectra_matches['map_y_y']) ** 2)
    spectra_matches_list.append(spectra_matches)
    #coord_matches = coord_df.iloc[id_matches, :].reset_index(drop=True)
    
spectra_matches_all = pd.concat(spectra_matches_list)
#spectra_matches_all = spectra_matches_all.query('distance<1').reset_index(drop=True)
spectra_matches_all#.query('distance<1').to_csv('spectra_matches.csv')

,map_x_x,map_y_x,id_x,wn_saturated,no_signal,id_file_x,feat_id,map_x_y,map_y_y,size,area,perimeter,circularity,id_y,id_file_y,distance
0,15507.299805,-786.000000,DC_passive_MarApr2024_C9_20240612_1,0,0,03 March 2024/DC_20240612,1,15507.150,-786.1769,1.057373,0.855008,3.585125,0.835714,03 March 2024/DC_20240612/DC_passive_MarApr202...,03 March 2024/DC_20240612,0.231808
1,15565.900391,-784.049988,DC_passive_MarApr2024_C9_20240612_2,0,0,03 March 2024/DC_20240612,2,15566.090,-784.2911,5.177734,19.919150,33.333090,0.225225,03 March 2024/DC_20240612/DC_passive_MarApr202...,03 March 2024/DC_20240612,0.306736
2,15556.049805,-771.400024,DC_passive_MarApr2024_C9_20240612_3,338,0,03 March 2024/DC_20240612,5,15556.270,-771.5998,10.923870,74.123260,61.633030,0.245145,03 March 2024/DC_20240612/DC_passive_MarApr202...,03 March 2024/DC_20240612,0.297315
3,15556.700195,-765.950012,DC_passive_MarApr2024_C9_20240612_4,1450,0,03 March 2024/DC_20240612,7,15556.680,-766.1406,1.647973,1.701550,5.622107,0.676306,03 March 2024/DC_20240612/DC_passive_MarApr202...,03 March 2024/DC_20240612,0.191655
4,15557.950195,-758.650024,DC_passive_MarApr2024_C9_20240612_5,1322,0,03 March 2024/DC_20240612,10,15557.950,-758.8942,0.712722,0.380944,2.328753,0.882492,03 March 2024/DC_20240612/DC_passive_MarApr202...,03 March 2024/DC_20240612,0.244176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,2936.699951,-1683.550049,ICB_passive_December2024_C2_20250121_209,0,0,12 December 2024/ICB_20250121,54,2936.717,-1683.5400,0.919229,0.558718,4.137100,0.410107,12 December 2024/ICB_20250121/ICB_passive_Dec2...,12 December 2024/ICB_20250121,0.019790
209,2957.949951,-1681.550049,ICB_passive_December2024_C2_20250121_210,0,0,12 December 2024/ICB_20250121,55,2957.927,-1681.5300,1.016188,0.778819,3.553557,0.774830,12 December 2024/ICB_20250121/ICB_passive_Dec2...,12 December 2024/ICB_20250121,0.030475
210,2886.000000,-1675.800049,ICB_passive_December2024_C2_20250121_211,0,0,12 December 2024/ICB_20250121,56,2886.000,-1675.7990,0.626147,0.296290,2.113194,0.833556,12 December 2024/ICB_20250121/ICB_passive_Dec2...,12 December 2024/ICB_20250121,0.001049
211,2972.750000,-1672.849976,ICB_passive_December2024_C2_20250121_212,0,0,12 December 2024/ICB_20250121,57,2972.763,-1672.8620,1.348818,1.354468,4.765286,0.749354,12 December 2024/ICB_20250121/ICB_passive_Dec2...,12 December 2024/ICB_20250121,0.017708


In [34]:
spectra_matches_all.query('distance>1')

,map_x_x,map_y_x,id_x,wn_saturated,no_signal,id_file_x,feat_id,map_x_y,map_y_y,size,area,perimeter,circularity,id_y,id_file_y,distance
60,20773.699219,-1105.050049,IAO_passive_MarApr2024_H9_20240625_61,0,0,03 March 2024/IAO_20240625,1,20772.910,-1105.8410,0.712722,0.380944,2.328753,0.882492,03 March 2024/IAO_20240625/IAO_passive_MarApr2...,03 March 2024/IAO_20240625,1.117350
37,24303.349609,3493.550049,IAO_passive_AprMay2024_I3_20240605_38,1450,0,04 April 2024/IAO_20240605,13,24303.180,3498.6800,8.492449,54.601990,37.564900,0.486117,04 April 2024/IAO_20240605/IAO_passive_AprMay2...,04 April 2024/IAO_20240605,5.132754
63,24329.000000,3636.800049,IAO_passive_AprMay2024_I3_20240605_64,0,0,04 April 2024/IAO_20240605,14,24327.910,3635.9370,0.581832,0.270894,1.992338,0.857372,04 April 2024/IAO_20240605/IAO_passive_AprMay2...,04 April 2024/IAO_20240605,1.390307
218,21981.550781,-2430.949951,FHS_passive_May2024_F10_20240709_219,0,1,05 May 2024/FHS_20240709,56,21981.810,-2429.8420,6.870202,43.944020,38.745050,0.367759,05 May 2024/FHS_20240709/FHS_passive_May2024_F...,05 May 2024/FHS_20240709,1.137871
131,-1608.400024,-2231.949951,ICB_passive_May2024_B8_20240716_132,150,0,05 May 2024/ICB_20240716,63,-1609.084,-2233.5460,19.457170,275.955900,135.874400,0.187785,05 May 2024/ICB_20240716/ICB_passive_May2024_B...,05 May 2024/ICB_20240716,1.736432
221,15848.099609,1898.750000,FHS_passive_June2024_C3_20240724_222,0,1,06 June 2024/FHS_20240724,20,15824.040,1904.3720,2.197823,3.572409,8.686826,0.594751,06 June 2024/FHS_20240724/FHS_passive_June2024...,06 June 2024/FHS_20240724,24.707725
78,16278.150391,1887.800049,ICB_passive_June2024_B4_20240731_79,0,1,06 June 2024/ICB_20240731,12,16264.840,1876.5560,1.284187,0.507926,7.017827,0.129566,06 June 2024/ICB_20240731/ICB_passive_June2024...,06 June 2024/ICB_20240731,17.423982
222,20488.900391,-1258.900024,DC_passive_July2024_I9_20240806_223,388,0,07 July 2024/DC_20240806,41,20489.020,-1247.9430,1.324653,1.219021,6.174081,0.401757,07 July 2024/DC_20240806/DC_passive_July2024_I...,07 July 2024/DC_20240806,10.957677
86,16229.150391,898.950012,FHS_passive_July2024_D6_20240807_51,0,0,07 July 2024/FHS_20240807,87,16228.860,901.2508,0.555371,0.008465,1.332584,0.059890,07 July 2024/FHS_20240807/FHS_passive_July2024...,07 July 2024/FHS_20240807,2.319041
87,16224.299805,907.549988,FHS_passive_July2024_D6_20240807_52,0,0,07 July 2024/FHS_20240807,92,16225.210,904.8261,0.525467,0.160843,1.960770,0.525588,07 July 2024/FHS_20240807/FHS_passive_July2024...,07 July 2024/FHS_20240807,2.871937


In [6]:
preposs_data_raman_new = preposs_data_raman.merge(spectra_matches_all.loc[:,['id_x','size','area','perimeter','circularity','distance']], 
                                          how='left', left_on="id", right_on="id_x").reset_index(drop=True)
preposs_data_raman_new.drop('id_x', axis=1).to_csv('passive_data_raman_specy_particle_info.csv')

# Active Data

## MEC_04_014

In [35]:
directory_name = 'MEC_04_014 (20240808 Daycroft Active Sampling)/Raw Data/OPTIR/' 

path = 'MEC_04_014 (20240808 Daycroft Active Sampling)/Raw Data/OPTIR/'

folders = os.listdir(path)
folders = [x for x in folders if os.path.isdir(path+x)]

files_list = []

for folder in folders:
    dirs = os.listdir(path+folder)
    files_list = files_list + [folder+'/'+x for x in dirs if x[-3:]=="csv"]
    for d in dirs:
        if os.path.isdir(path+folder+'/'+d):
            dirs2 = os.listdir(path+folder+'/'+d)
            files_list = files_list + [folder+'/'+d+'/'+y for y in dirs2 if y[-3:]=="csv"]
files_list

['Stage 4/DC_Active_20240808_S4_GridSi_FOV1_20241009.csv',
 'Stage 4/DC_Active_20240808_S4_GridSi_FOV1_20240911.csv',
 'Stage 3/DC_Active_20240808_S3_GridSi_FOV1_20241001.csv',
 'Stage 3/DC_Active_20240808_S3_GridSi_FOV1_20241002.csv',
 'Stage 3/DC_Active_20240808_S3_GridSi_FOV1_20241010.csv',
 'Stage 3/DC_Active_20240808_S3_GridSi_FOV1_20241011.csv',
 'Stage 3/DC_Active_20240808_S3_GridSi_FOV2_20241011.csv',
 'Stage 2/20241210/MEC_04_014_DC_Active_20240808_S2_GridSi_20241210_FOV14.csv',
 'Stage 2/20241210/MEC_04_014_DC_Active_20240808_S2_GridSi_20241210_FOV28.csv',
 'Stage 2/20241210/MEC_04_014_DC_Active_20240808_S2_GridSi_20241210_FOV29.csv',
 'Stage 2/20241210/MEC_04_014_DC_Active_20240808_S2_GridSi_20241210_FOV15.csv',
 'Stage 2/20241210/MEC_04_014_DC_Active_20240808_S2_GridSi_20241210_FOV17.csv',
 'Stage 2/20241210/MEC_04_014_DC_Active_20240808_S2_GridSi_20241210_FOV16.csv',
 'Stage 2/20241210/MEC_04_014_DC_Active_20240808_S2_GridSi_20241210_FOV12.csv',
 'Stage 2/20241210/MEC_04_0

In [36]:
df_list = []
prev_mosaic = ''
count = 1
for f in files_list:
    df = pd.read_csv(path + f)
    df.columns = ['feat_id', 'map_x', 'map_y', 'size', 'area', 'perimeter', 'circularity']
    mosaic = '/'.join(f.split('/')[:-1])
    #mosaic = '/'.join(f.split('/')[:-1])
    #mosaic = f[:-4]
    if mosaic == prev_mosaic:
        count = count + 1
    df['id'] = [f[:-4] + '_' +
                str(i + 1) for i in range(df.shape[0])]

    df['id_file'] = [mosaic for i in range(df.shape[0])]
    
    #df['brand'] = files_codes.loc[f, 'brand']
    #df['run'] = files_codes.loc[f, 'run']
    #df['mosaic'] = files_codes.loc[f, 'mosaic']
    df_list.append(df.iloc[:,:])
    #prev_mosaic = mosaic
    
coord_df = pd.concat(df_list,axis=0).reset_index(drop=True)
coord_df

,feat_id,map_x,map_y,size,area,perimeter,circularity,id,id_file
0,1,11350.370,-2681.947,2.387170,4.554399,8.426622,0.805787,Stage 4/DC_Active_20240808_S4_GridSi_FOV1_2024...,Stage 4
1,2,11330.930,-2682.033,2.920689,6.603032,11.091790,0.674274,Stage 4/DC_Active_20240808_S4_GridSi_FOV1_2024...,Stage 4
2,3,11313.780,-2681.414,2.390522,4.571330,9.162588,0.684073,Stage 4/DC_Active_20240808_S4_GridSi_FOV1_2024...,Stage 4
3,4,11364.290,-2680.864,1.626374,2.133287,6.326505,0.669604,Stage 4/DC_Active_20240808_S4_GridSi_FOV1_2024...,Stage 4
4,5,11417.460,-2679.240,2.835288,6.179760,11.504420,0.586596,Stage 4/DC_Active_20240808_S4_GridSi_FOV1_2024...,Stage 4
...,...,...,...,...,...,...,...,...,...
1120,38,9761.176,-2226.253,1.714811,2.201010,7.062470,0.554377,Stage 6/DC_Active_20240808_S6_ThinSi_FOV1_2024...,Stage 6
1121,39,9738.252,-2221.877,1.665275,2.048633,6.281861,0.652205,Stage 6/DC_Active_20240808_S6_ThinSi_FOV1_2024...,Stage 6
1122,40,9783.782,-2221.775,3.472786,9.218847,12.900140,0.695960,Stage 6/DC_Active_20240808_S6_ThinSi_FOV1_2024...,Stage 6
1123,41,9750.866,-2219.030,2.268131,3.978750,8.839249,0.639753,Stage 6/DC_Active_20240808_S6_ThinSi_FOV1_2024...,Stage 6


In [37]:
preposs_data_raman = pd.read_csv('active_data_MEC_04_014_raman_specy.csv')
del preposs_data_raman['Unnamed: 0']
preposs_data_raman

,600.0,602.0,604.0,606.0,608.0,610.0,612.0,614.0,616.0,618.0,...,3492.0,3494.0,3496.0,3498.0,map_x,map_y,id,wn_saturated,no_signal,id_file
0,-0.268626,-0.239331,-0.202112,-0.151761,-0.113264,-0.097943,-0.097373,-0.106532,-0.123398,-0.144607,...,-0.268626,-0.268626,-0.268626,-0.268626,11350.549805,-2681.649902,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,0,0,Stage 4
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,11330.700195,-2682.000000,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,1450,0,Stage 4
2,-0.320248,-0.320248,-0.320248,-0.320248,-0.320248,-0.320248,-0.320248,-0.320248,-0.273819,-0.224512,...,-0.320248,-0.320248,-0.320248,-0.320248,11313.599609,-2681.750000,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,0,0,Stage 4
3,-0.261241,-0.261241,-0.261241,-0.261241,-0.261241,-0.261241,-0.229514,-0.176862,-0.122369,-0.072778,...,-0.261241,-0.261241,-0.261241,-0.261241,11364.099609,-2680.850098,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,0,0,Stage 4
4,-0.319058,-0.245167,-0.157064,-0.107878,-0.051101,0.020514,0.119441,0.236381,0.377897,0.516118,...,-0.319058,-0.319058,-0.319058,-0.319058,11417.250000,-2678.949951,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,1149,0,Stage 4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1188,-0.109944,-0.088083,-0.066089,-0.043769,-0.024065,-0.006465,0.000520,0.003603,-0.010531,-0.029513,...,-0.163847,-0.163847,-0.163847,-0.163847,211.550003,-3675.500000,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,0,0,Stage 1
1189,-0.163496,-0.130761,-0.097557,-0.063204,-0.034733,-0.010997,0.004998,0.018423,0.022383,0.021957,...,-0.180057,-0.180057,-0.180057,-0.180057,964.049988,-3626.500000,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,0,0,Stage 1
1190,-0.501878,-0.501878,-0.501878,-0.501878,-0.479260,-0.438438,-0.408108,-0.376406,-0.303426,-0.211123,...,-0.501878,-0.501878,-0.501878,-0.501878,940.349976,-3691.449951,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,7,0,Stage 1
1191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1046.050049,-3716.949951,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,1450,0,Stage 1


In [40]:
spectra_matches_list = []
for id_ in np.unique(preposs_data_raman['id_file']):
    
    if coord_df.query('id_file == @id_').shape[0] == 0:
        continue
    id_matches = np.argmin(pairwise_distances(preposs_data_raman.query('id_file == @id_').loc[:,['map_x', 'map_y']], 
                                              coord_df.query('id_file == @id_').loc[:,['map_x', 'map_y']]),1)


    coord_matches = coord_df.query('id_file == @id_').iloc[id_matches, :].reset_index(drop=True)
    spectra_matches = (preposs_data_raman.
                       query('id_file == @id_').
                       reset_index(drop=True).
                       merge(coord_matches, left_index=True, right_index=True).iloc[:, -15:])
    
    spectra_matches['distance'] = np.sqrt((spectra_matches['map_x_x'] - spectra_matches['map_x_y']) ** 2 + 
                                          (spectra_matches['map_y_x'] - spectra_matches['map_y_y']) ** 2)
    spectra_matches_list.append(spectra_matches)
    #coord_matches = coord_df.iloc[id_matches, :].reset_index(drop=True)
    
spectra_matches_all = pd.concat(spectra_matches_list)
spectra_matches_all = spectra_matches_all.query('distance<1').reset_index(drop=True)
spectra_matches_all#.query('distance<1').to_csv('spectra_matches.csv')

,map_x_x,map_y_x,id_x,wn_saturated,no_signal,id_file_x,feat_id,map_x_y,map_y_y,size,area,perimeter,circularity,id_y,id_file_y,distance
0,9739.200195,-797.250000,MEC_04_014_DC_Active_20240808_S3_GridSi_202410...,327,0,0.5% Mistakes,9,9739.315,-797.4259,3.059988,7.237938,10.387390,0.842749,0.5% Mistakes/DC_Active_20240808_S3_GridSi_FOV...,0.5% Mistakes,0.210050
1,9688.500000,-790.450012,MEC_04_014_DC_Active_20240808_S3_GridSi_202410...,1450,0,0.5% Mistakes,11,9688.419,-790.7115,3.492868,9.675981,12.316590,0.801328,0.5% Mistakes/DC_Active_20240808_S3_GridSi_FOV...,0.5% Mistakes,0.273746
2,9778.099609,-789.349976,MEC_04_014_DC_Active_20240808_S3_GridSi_202410...,501,0,0.5% Mistakes,12,9778.130,-789.5494,7.389710,38.026690,29.182910,0.560954,0.5% Mistakes/DC_Active_20240808_S3_GridSi_FOV...,0.5% Mistakes,0.201727
3,9739.500000,-785.349976,MEC_04_014_DC_Active_20240808_S3_GridSi_202410...,0,0,0.5% Mistakes,14,9739.356,-785.5985,3.663777,10.344750,12.608370,0.817521,0.5% Mistakes/DC_Active_20240808_S3_GridSi_FOV...,0.5% Mistakes,0.287229
4,9702.450195,-783.450012,MEC_04_014_DC_Active_20240808_S3_GridSi_202410...,0,0,0.5% Mistakes,15,9702.453,-783.7190,3.977582,11.885460,13.927870,0.769737,0.5% Mistakes/DC_Active_20240808_S3_GridSi_FOV...,0.5% Mistakes,0.269002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984,14611.849609,-1758.699951,MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...,0,0,Stage 6,43,14611.740,-1758.9550,1.780224,2.471904,6.434285,0.750114,Stage 6/DC_Active_20240808_S6_ThinSi_FOV3_2024...,Stage 6,0.277604
985,14604.000000,-1757.150024,MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...,0,0,Stage 6,44,14604.180,-1757.4350,1.736788,2.336457,6.326505,0.733376,Stage 6/DC_Active_20240808_S6_ThinSi_FOV3_2024...,Stage 6,0.337062
986,14621.700195,-1756.500000,MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...,0,0,Stage 6,45,14621.400,-1756.6070,1.788476,2.429577,6.878479,0.645123,Stage 6/DC_Active_20240808_S6_ThinSi_FOV3_2024...,Stage 6,0.318695
987,14688.099609,-1754.650024,MEC_04_014_DC_Active_20240808_S6_ThinSi_202410...,0,0,Stage 6,46,14688.310,-1754.8920,1.373244,1.388330,6.174081,0.457556,Stage 6/DC_Active_20240808_S6_ThinSi_FOV3_2024...,Stage 6,0.320650


In [41]:
preposs_data_raman_new = preposs_data_raman.merge(spectra_matches_all.loc[:,['id_x','size','area','perimeter','circularity','distance']], 
                                          how='left', left_on="id", right_on="id_x").reset_index(drop=True)
preposs_data_raman_new.drop('id_x', axis=1)#.to_csv('active_data_MEC_04_014_raman_specy_particle_info.csv')


,600.0,602.0,604.0,606.0,608.0,610.0,612.0,614.0,616.0,618.0,...,map_y,id,wn_saturated,no_signal,id_file,size,area,perimeter,circularity,distance
0,-0.268626,-0.239331,-0.202112,-0.151761,-0.113264,-0.097943,-0.097373,-0.106532,-0.123398,-0.144607,...,-2681.649902,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,0,0,Stage 4,2.387170,4.554399,8.426622,0.805787,0.347270
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-2682.000000,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,1450,0,Stage 4,2.920689,6.603032,11.091790,0.674274,0.232162
2,-0.320248,-0.320248,-0.320248,-0.320248,-0.320248,-0.320248,-0.320248,-0.320248,-0.273819,-0.224512,...,-2681.750000,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,0,0,Stage 4,2.390522,4.571330,9.162588,0.684073,0.381362
3,-0.261241,-0.261241,-0.261241,-0.261241,-0.261241,-0.261241,-0.229514,-0.176862,-0.122369,-0.072778,...,-2680.850098,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,0,0,Stage 4,1.626374,2.133287,6.326505,0.669604,0.190898
4,-0.319058,-0.245167,-0.157064,-0.107878,-0.051101,0.020514,0.119441,0.236381,0.377897,0.516118,...,-2678.949951,MEC_04_014_DC_Active_20240808_S4_GridSi_202410...,1149,0,Stage 4,2.835288,6.179760,11.504420,0.586596,0.358090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1188,-0.109944,-0.088083,-0.066089,-0.043769,-0.024065,-0.006465,0.000520,0.003603,-0.010531,-0.029513,...,-3675.500000,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,0,0,Stage 1,NaN,NaN,NaN,NaN,NaN
1189,-0.163496,-0.130761,-0.097557,-0.063204,-0.034733,-0.010997,0.004998,0.018423,0.022383,0.021957,...,-3626.500000,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,0,0,Stage 1,NaN,NaN,NaN,NaN,NaN
1190,-0.501878,-0.501878,-0.501878,-0.501878,-0.479260,-0.438438,-0.408108,-0.376406,-0.303426,-0.211123,...,-3691.449951,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,7,0,Stage 1,NaN,NaN,NaN,NaN,NaN
1191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-3716.949951,MEC_04_014_DC_Active_20240808_S1_GridSi_202503...,1450,0,Stage 1,NaN,NaN,NaN,NaN,NaN


In [44]:
np.where(np.isnan(preposs_data_raman_new['distance']))[0].shape

(204,)

## MEC_04_023

In [45]:
directory_name = 'MEC_04_023 (Daycroft Active Sampling 20241212)/Raw Data/OPTIR/' 
path = 'MEC_04_023 (Daycroft Active Sampling 20241212)/Raw Data/OPTIR/'

folders = os.listdir(path)
folders = [x for x in folders if os.path.isdir(path+x)]

files_list = []

for folder in folders:
    dirs = os.listdir(path+folder)
    
    files_list = files_list + [folder+'/'+x for x in dirs if x[-3:]=="csv"]
    for d in dirs:
        if os.path.isdir(path+folder+'/'+d):
            dirs2 = os.listdir(path+folder+'/'+d)
            files_list = files_list + [folder+'/'+d+'/'+y for y in dirs2 if y[-3:]=="csv"]
files_list


['S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV8.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV9.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV2.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV3.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV11.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV1.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV10.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV4.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV5.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV7.csv',
 'S2/20250221/MEC_04_023_DC_Active_20241212_S2_GridSi_20250221_FOV6.csv',
 'S2/20250220/MEC_04_023_DC_Active_20241212_S2_GridSi_20250220_FOV8.csv',
 'S2/20250220/MEC_04_023_DC_Active_20241212_S2_GridSi_20250220_FOV9.csv',
 'S2/20250220/MEC_04_023_DC_Active_2

In [46]:
df_list = []
prev_mosaic = ''
count = 1
for f in files_list:
    df = pd.read_csv(path + f)
    df.columns = ['feat_id', 'map_x', 'map_y', 'size', 'area', 'perimeter', 'circularity']
    #mosaic = files_codes.loc[f, 'mosaic']
    mosaic = '/'.join(f.split('/')[:-1])
    #mosaic = f[:-4]
    if mosaic == prev_mosaic:
        count = count + 1
    df['id'] = [f[:-4] + '_' +
                str(i + 1) for i in range(df.shape[0])]
    
    #df['id'] = [files_codes.loc[f, 'brand'] + '_' + 
    #            files_codes.loc[f, 'material'] + '_' + 
    #            files_codes.loc[f, 'run'] + '_' + 
    #            files_codes.loc[f, 'mosaic'] + '_' +
    #            str(count) + '_' +
    #            str(i + 1) for i in range(df.shape[0])]
    
    df['id_file'] = [mosaic for i in range(df.shape[0])]
    
    #df['brand'] = files_codes.loc[f, 'brand']
    #df['run'] = files_codes.loc[f, 'run']
    #df['mosaic'] = files_codes.loc[f, 'mosaic']
    df_list.append(df.iloc[:,:])
    #prev_mosaic = mosaic
    
coord_df = pd.concat(df_list,axis=0).reset_index(drop=True)
coord_df

,feat_id,map_x,map_y,size,area,perimeter,circularity,id,id_file
0,1,12362.650,-5839.375,7.260091,41.379000,28.339170,0.647295,S2/20250221/MEC_04_023_DC_Active_20241212_S2_G...,S2/20250221
1,2,12361.500,-5787.165,3.716739,9.752170,14.492920,0.583291,S2/20250221/MEC_04_023_DC_Active_20241212_S2_G...,S2/20250221
2,3,12387.690,-5741.978,5.077993,19.682110,18.033410,0.760348,S2/20250221/MEC_04_023_DC_Active_20241212_S2_G...,S2/20250221
3,1,11427.250,-5983.032,1.165179,0.948128,3.921540,0.774551,S2/20250221/MEC_04_023_DC_Active_20241212_S2_G...,S2/20250221
4,2,11488.560,-5977.258,1.111480,0.973524,3.997752,0.765264,S2/20250221/MEC_04_023_DC_Active_20241212_S2_G...,S2/20250221
...,...,...,...,...,...,...,...,...,...
856,127,8123.284,3315.013,2.593957,5.350149,10.463600,0.613903,S6/20250131/MEC_04_023_DC_Active_20241212_S6_t...,S6/20250131
857,128,8195.579,3316.057,1.007226,0.744957,4.244879,0.519394,S6/20250131/MEC_04_023_DC_Active_20241212_S6_t...,S6/20250131
858,129,8199.822,3316.367,0.516043,0.152378,2.512744,0.303194,S6/20250131/MEC_04_023_DC_Active_20241212_S6_t...,S6/20250131
859,130,8154.678,3316.417,0.900497,0.626441,3.185574,0.775535,S6/20250131/MEC_04_023_DC_Active_20241212_S6_t...,S6/20250131


In [47]:
np.unique(coord_df['id_file'])

array(['S1', 'S2/20250207', 'S2/20250220', 'S2/20250221', 'S3/20250122',
       'S3/20250130', 'S4/20241218', 'S6/20250130', 'S6/20250131'],
      dtype=object)

In [48]:
preposs_data_raman = pd.read_csv('active_data_MEC_04_023_raman_specy.csv')
del preposs_data_raman['Unnamed: 0']
preposs_data_raman

,600.0,602.0,604.0,606.0,608.0,610.0,612.0,614.0,616.0,618.0,...,3492.0,3494.0,3496.0,3498.0,map_x,map_y,id,wn_saturated,no_signal,id_file
0,-0.428775,-0.427159,-0.423753,-0.415965,-0.387402,-0.342117,-0.296173,-0.249350,-0.194346,-0.158488,...,-0.632023,-0.632023,-0.632023,-0.632023,9691.349609,-3860.500000,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,443,0,S2/20250221
1,-0.490299,-0.493227,-0.487824,-0.462022,-0.451574,-0.453481,-0.388258,-0.312751,-0.288921,-0.271999,...,-0.736022,-0.736022,-0.736022,-0.736022,9717.500000,-3835.699951,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,388,0,S2/20250221
2,-0.292037,-0.292037,-0.279882,-0.237964,-0.191629,-0.141741,-0.099381,-0.058089,-0.010041,0.023814,...,-0.292037,-0.292037,-0.292037,-0.292037,9627.950195,-3799.750000,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,0,0,S2/20250221
3,-0.148054,-0.133265,-0.112935,-0.079038,-0.047900,-0.018982,0.000896,0.017704,0.022677,0.022810,...,-0.181043,-0.181043,-0.181043,-0.181043,9667.950195,-3773.000000,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,0,0,S2/20250221
4,3.819011,3.306662,2.904541,2.772344,2.619763,2.450776,2.236700,2.019254,1.875956,1.736015,...,-0.270720,-0.269543,-0.268374,-0.267219,9613.250000,-3759.800049,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,1393,0,S2/20250221
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,-0.152106,-0.140476,-0.122895,-0.090740,-0.065336,-0.045364,-0.033441,-0.023919,-0.021230,-0.023598,...,-0.181947,-0.181947,-0.181947,-0.181947,8084.250000,3314.300049,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131
949,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,...,-0.237548,-0.237548,-0.237548,-0.237548,8145.899902,3314.449951,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131
950,-0.160381,-0.120713,-0.090451,-0.083226,-0.069407,-0.050281,-0.032653,-0.016661,-0.015192,-0.016567,...,-0.176546,-0.176546,-0.176546,-0.176546,8093.200195,3314.649902,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131
951,-0.295109,-0.295109,-0.282537,-0.239179,-0.208552,-0.188170,-0.169163,-0.152514,-0.158744,-0.172089,...,-0.295109,-0.295109,-0.295109,-0.295109,8213.900391,3315.000000,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131


In [49]:
spectra_matches_list = []
for id_ in np.unique(preposs_data_raman['id_file']):
    
    if coord_df.query('id_file == @id_').shape[0] == 0:
        continue
    id_matches = np.argmin(pairwise_distances(preposs_data_raman.query('id_file == @id_').loc[:,['map_x', 'map_y']], 
                                              coord_df.query('id_file == @id_').loc[:,['map_x', 'map_y']]),1)


    coord_matches = coord_df.query('id_file == @id_').iloc[id_matches, :].reset_index(drop=True)
    spectra_matches = (preposs_data_raman.
                       query('id_file == @id_').
                       reset_index(drop=True).
                       merge(coord_matches, left_index=True, right_index=True).iloc[:, -15:])
    
    spectra_matches['distance'] = np.sqrt((spectra_matches['map_x_x'] - spectra_matches['map_x_y']) ** 2 + 
                                          (spectra_matches['map_y_x'] - spectra_matches['map_y_y']) ** 2)
    spectra_matches_list.append(spectra_matches)
    #coord_matches = coord_df.iloc[id_matches, :].reset_index(drop=True)
    
spectra_matches_all = pd.concat(spectra_matches_list)
spectra_matches_all = spectra_matches_all.query('distance<1').reset_index(drop=True)
spectra_matches_all#.query('id_file_y=="S1"')#.query('distance<1').to_csv('spectra_matches.csv')

,map_x_x,map_y_x,id_x,wn_saturated,no_signal,id_file_x,feat_id,map_x_y,map_y_y,size,area,perimeter,circularity,id_y,id_file_y,distance
0,8389.400391,-6399.200195,MEC_04_023_DC_Active_20241212_S1_GridSi_202502...,224,0,S1,6,8389.480,-6398.931,13.325100,92.112290,111.436100,0.093189,S1/MEC_04_023_DC_Active_20241212_S1_GridSi_202...,S1,0.280720
1,8300.450195,-6357.250000,MEC_04_023_DC_Active_20241212_S1_GridSi_202502...,537,0,S1,14,8300.333,-6357.026,6.371856,16.727680,50.622870,0.082005,S1/MEC_04_023_DC_Active_20241212_S1_GridSi_202...,S1,0.252806
2,8528.950195,-6206.500000,MEC_04_023_DC_Active_20241212_S1_GridSi_202502...,417,0,S1,5,8528.920,-6206.247,14.898690,158.650500,78.276130,0.325296,S1/MEC_04_023_DC_Active_20241212_S1_GridSi_202...,S1,0.254796
3,13026.200195,-5103.450195,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,0,0,S2/20250207,1,13026.110,-5103.637,5.841700,25.557120,22.304440,0.645395,S2/20250207/MEC_04_023_DC_Active_20241212_S2_G...,S2/20250207,0.207440
4,13105.650391,-5066.899902,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,1450,0,S2/20250207,3,13105.840,-5066.667,6.039393,27.309460,25.387650,0.532310,S2/20250207/MEC_04_023_DC_Active_20241212_S2_G...,S2/20250207,0.300325
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,8084.250000,3314.300049,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,123,8084.061,3314.339,0.845570,0.541787,2.817591,0.857372,S6/20250131/MEC_04_023_DC_Active_20241212_S6_t...,S6/20250131,0.192972
748,8145.899902,3314.449951,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,124,8146.150,3314.476,1.045296,0.829612,3.661337,0.777485,S6/20250131/MEC_04_023_DC_Active_20241212_S6_t...,S6/20250131,0.251451
749,8093.200195,3314.649902,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,125,8092.994,3314.726,0.581321,0.262428,1.960770,0.857538,S6/20250131/MEC_04_023_DC_Active_20241212_S6_t...,S6/20250131,0.219789
750,8213.900391,3315.000000,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,126,8214.074,3315.092,0.634436,0.228567,2.956939,0.328416,S6/20250131/MEC_04_023_DC_Active_20241212_S6_t...,S6/20250131,0.196480


In [50]:
preposs_data_raman_new = preposs_data_raman.merge(spectra_matches_all.loc[:,['id_x','size','area','perimeter','circularity','distance']], 
                                          how='left', left_on="id", right_on="id_x").reset_index(drop=True)
preposs_data_raman_new.drop('id_x', axis=1)#.to_csv('active_data_MEC_04_023_raman_specy_particle_info.csv')


,600.0,602.0,604.0,606.0,608.0,610.0,612.0,614.0,616.0,618.0,...,map_y,id,wn_saturated,no_signal,id_file,size,area,perimeter,circularity,distance
0,-0.428775,-0.427159,-0.423753,-0.415965,-0.387402,-0.342117,-0.296173,-0.249350,-0.194346,-0.158488,...,-3860.500000,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,443,0,S2/20250221,5.940442,27.182480,28.173670,0.430228,0.330679
1,-0.490299,-0.493227,-0.487824,-0.462022,-0.451574,-0.453481,-0.388258,-0.312751,-0.288921,-0.271999,...,-3835.699951,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,388,0,S2/20250221,6.988038,36.384400,28.268370,0.572018,0.290274
2,-0.292037,-0.292037,-0.279882,-0.237964,-0.191629,-0.141741,-0.099381,-0.058089,-0.010041,0.023814,...,-3799.750000,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,0,0,S2/20250221,6.647790,32.473370,26.967360,0.560979,0.279725
3,-0.148054,-0.133265,-0.112935,-0.079038,-0.047900,-0.018982,0.000896,0.017704,0.022677,0.022810,...,-3773.000000,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,0,0,S2/20250221,1.617244,1.938582,6.465852,0.582544,0.300815
4,3.819011,3.306662,2.904541,2.772344,2.619763,2.450776,2.236700,2.019254,1.875956,1.736015,...,-3759.800049,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,1393,0,S2/20250221,4.967517,18.412300,19.397560,0.614767,0.398620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,-0.152106,-0.140476,-0.122895,-0.090740,-0.065336,-0.045364,-0.033441,-0.023919,-0.021230,-0.023598,...,3314.300049,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,0.845570,0.541787,2.817591,0.857372,0.192972
949,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,...,3314.449951,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,1.045296,0.829612,3.661337,0.777485,0.251451
950,-0.160381,-0.120713,-0.090451,-0.083226,-0.069407,-0.050281,-0.032653,-0.016661,-0.015192,-0.016567,...,3314.649902,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,0.581321,0.262428,1.960770,0.857538,0.219789
951,-0.295109,-0.295109,-0.282537,-0.239179,-0.208552,-0.188170,-0.169163,-0.152514,-0.158744,-0.172089,...,3315.000000,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,0.634436,0.228567,2.956939,0.328416,0.196480


In [54]:
preposs_data_raman_new

,600.0,602.0,604.0,606.0,608.0,610.0,612.0,614.0,616.0,618.0,...,id,wn_saturated,no_signal,id_file,id_x,size,area,perimeter,circularity,distance
0,-0.428775,-0.427159,-0.423753,-0.415965,-0.387402,-0.342117,-0.296173,-0.249350,-0.194346,-0.158488,...,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,443,0,S2/20250221,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,5.940442,27.182480,28.173670,0.430228,0.330679
1,-0.490299,-0.493227,-0.487824,-0.462022,-0.451574,-0.453481,-0.388258,-0.312751,-0.288921,-0.271999,...,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,388,0,S2/20250221,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,6.988038,36.384400,28.268370,0.572018,0.290274
2,-0.292037,-0.292037,-0.279882,-0.237964,-0.191629,-0.141741,-0.099381,-0.058089,-0.010041,0.023814,...,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,0,0,S2/20250221,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,6.647790,32.473370,26.967360,0.560979,0.279725
3,-0.148054,-0.133265,-0.112935,-0.079038,-0.047900,-0.018982,0.000896,0.017704,0.022677,0.022810,...,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,0,0,S2/20250221,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,1.617244,1.938582,6.465852,0.582544,0.300815
4,3.819011,3.306662,2.904541,2.772344,2.619763,2.450776,2.236700,2.019254,1.875956,1.736015,...,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,1393,0,S2/20250221,MEC_04_023_DC_Active_20241212_S2_GridSi_202502...,4.967517,18.412300,19.397560,0.614767,0.398620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,-0.152106,-0.140476,-0.122895,-0.090740,-0.065336,-0.045364,-0.033441,-0.023919,-0.021230,-0.023598,...,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0.845570,0.541787,2.817591,0.857372,0.192972
949,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,-0.237548,...,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,1.045296,0.829612,3.661337,0.777485,0.251451
950,-0.160381,-0.120713,-0.090451,-0.083226,-0.069407,-0.050281,-0.032653,-0.016661,-0.015192,-0.016567,...,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0.581321,0.262428,1.960770,0.857538,0.219789
951,-0.295109,-0.295109,-0.282537,-0.239179,-0.208552,-0.188170,-0.169163,-0.152514,-0.158744,-0.172089,...,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0,0,S6/20250131,MEC_04_023_DC_Active_20241212_S6_ThinSi_202501...,0.634436,0.228567,2.956939,0.328416,0.196480


In [53]:
np.where(np.isnan(preposs_data_raman_new['distance']))[0].shape

(201,)

## MEC_02_119

In [19]:
directory_name = 'MEC_02_119 (Active Sampling at DC 20240418)/Raw Data/OPTIR/' 

path = 'MEC_02_119 (Active Sampling at DC 20240418)/Raw Data/OPTIR/'

folders = os.listdir(path)
folders = [x for x in folders if os.path.isdir(path+x)]

files_list = []

for folder in folders:
    dirs = os.listdir(path+folder)
    
    files_list = files_list + [folder+'/'+x for x in dirs if x[-3:]=="csv"]
    for d in dirs:
        if os.path.isdir(path+folder+'/'+d):
            dirs2 = os.listdir(path+folder+'/'+d)
            files_list = files_list + [folder+'/'+d+'/'+y for y in dirs2 if y[-3:]=="csv"]
files_list


['S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV4.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV5.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV7.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV6.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV2.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV3.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20240522.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV21.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV20.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV22.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV23.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV24.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV18.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV19.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV25.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV14.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV15.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV17.csv',
 'S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV

In [20]:
df_list = []
prev_mosaic = ''
count = 1
for f in files_list:
    df = pd.read_csv(path + f)
    df.columns = ['feat_id', 'map_x', 'map_y', 'size', 'area', 'perimeter', 'circularity']
    #mosaic = files_codes.loc[f, 'mosaic']
    mosaic = '/'.join(f.split('/')[:-1])
    #mosaic = f[:-4]
    if mosaic == prev_mosaic:
        count = count + 1
    df['id'] = [f[:-4] + '_' +
                str(i + 1) for i in range(df.shape[0])]
    
    #df['id'] = [files_codes.loc[f, 'brand'] + '_' + 
    #            files_codes.loc[f, 'material'] + '_' + 
    #            files_codes.loc[f, 'run'] + '_' + 
    #            files_codes.loc[f, 'mosaic'] + '_' +
    #            str(count) + '_' +
    #            str(i + 1) for i in range(df.shape[0])]
    
    df['id_file'] = [mosaic for i in range(df.shape[0])]
    
    #df['brand'] = files_codes.loc[f, 'brand']
    #df['run'] = files_codes.loc[f, 'run']
    #df['mosaic'] = files_codes.loc[f, 'mosaic']
    df_list.append(df.iloc[:,:])
    #prev_mosaic = mosaic
    
coord_df = pd.concat(df_list,axis=0).reset_index(drop=True)
coord_df

,feat_id,map_x,map_y,size,area,perimeter,circularity,id,id_file
0,1,16442.65,-5546.0090,0.648256,0.313221,2.436533,0.662830,S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV4_1,S2/20241203
1,2,16443.89,-5544.1920,0.521868,0.169309,1.624355,0.806145,S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV4_2,S2/20241203
2,3,16442.93,-5544.4090,5.494526,20.647170,22.520000,0.511470,S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV4_3,S2/20241203
3,4,16399.58,-5517.9580,5.298066,22.297930,18.934870,0.781333,S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV4_4,S2/20241203
4,5,16452.28,-5513.2430,7.679178,46.297410,27.500840,0.769062,S2/20241203/MEC_02_119_DC_Sp_S2_GridSi_FOV4_5,S2/20241203
...,...,...,...,...,...,...,...,...,...
1171,72,18853.11,214.5450,1.090201,0.957468,3.770599,0.845950,S6/AlSi/MEC_02_119_DC_Sp_S6_SquareAlSi_FOV2_20...,S6/AlSi
1172,73,18875.01,215.0112,0.872364,0.591004,3.126375,0.759536,S6/AlSi/MEC_02_119_DC_Sp_S6_SquareAlSi_FOV2_20...,S6/AlSi
1173,74,18861.08,215.3535,0.902293,0.603713,2.996222,0.844741,S6/AlSi/MEC_02_119_DC_Sp_S6_SquareAlSi_FOV2_20...,S6/AlSi
1174,75,18891.78,215.9880,0.691888,0.368583,2.612305,0.678466,S6/AlSi/MEC_02_119_DC_Sp_S6_SquareAlSi_FOV2_20...,S6/AlSi


In [21]:
preposs_data_raman = pd.read_csv('active_data_MEC_02_119_raman_specy.csv')
del preposs_data_raman['Unnamed: 0']
preposs_data_raman

,600.0,602.0,604.0,606.0,608.0,610.0,612.0,614.0,616.0,618.0,...,3492.0,3494.0,3496.0,3498.0,map_x,map_y,id,wn_saturated,no_signal,id_file
0,-0.241803,-0.345840,-0.447513,-0.267926,-0.129066,-0.069833,0.004492,0.088770,0.127176,0.143486,...,-0.460840,-0.460840,-0.460840,-0.460840,15800.549805,-6182.600098,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_1,0,0,S2/20241203
1,-0.079612,-0.079612,-0.079612,-0.079612,-0.079612,-0.079612,-0.079612,-0.079612,-0.079612,-0.079612,...,-0.079612,-0.079612,-0.079612,-0.079612,15881.900391,-6173.350098,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_2,0,1,S2/20241203
2,-0.068228,-0.068228,-0.068228,-0.068228,-0.068228,-0.068228,-0.068228,-0.068228,-0.068228,-0.068228,...,-0.068228,-0.068228,-0.068228,-0.068228,15872.900391,-6172.950195,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_3,0,1,S2/20241203
3,-0.272297,-0.272297,-0.272297,-0.272297,-0.272297,-0.272297,-0.229919,-0.159594,-0.134822,-0.119084,...,-0.272297,-0.272297,-0.272297,-0.272297,15878.250000,-6166.600098,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_4,0,0,S2/20241203
4,-0.390401,-0.390401,-0.390401,-0.390401,-0.390401,-0.390401,-0.344589,-0.268565,-0.226251,-0.193516,...,-0.390401,-0.390401,-0.390401,-0.390401,15875.799805,-6166.799805,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20241203_5,0,0,S2/20241203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1170,-0.209100,-0.248611,-0.276065,-0.295087,-0.300068,-0.279036,-0.132708,-0.029530,0.020243,0.017097,...,-0.785205,-0.785205,-0.785205,-0.785205,18857.199219,213.500000,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_130,0,0,S6/AlSi
1171,-0.590409,-0.595824,-0.578345,-0.544856,-0.497415,-0.455610,-0.453678,-0.466968,-0.499099,-0.520913,...,-0.751973,-0.751973,-0.751973,-0.751973,18852.800781,214.750000,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_131,0,0,S6/AlSi
1172,-0.440795,-0.431276,-0.439299,-0.459590,-0.479026,-0.495371,-0.496224,-0.486509,-0.463715,-0.447584,...,-0.605988,-0.605988,-0.605988,-0.605988,18875.300781,214.850006,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_132,0,0,S6/AlSi
1173,-0.575353,-0.712964,-0.769593,-0.769593,-0.682794,-0.596267,-0.576109,-0.571957,-0.587614,-0.593781,...,-0.769593,-0.769593,-0.769593,-0.769593,18860.849609,215.250000,MEC_02_119_DC_Sp_S6_SquareAlSi_20240502_133,0,0,S6/AlSi


In [22]:
spectra_matches_list = []
for id_ in np.unique(preposs_data_raman['id_file']):
    
    if coord_df.query('id_file == @id_').shape[0] == 0:
        continue
    id_matches = np.argmin(pairwise_distances(preposs_data_raman.query('id_file == @id_').loc[:,['map_x', 'map_y']], 
                                              coord_df.query('id_file == @id_').loc[:,['map_x', 'map_y']]),1)


    coord_matches = coord_df.query('id_file == @id_').iloc[id_matches, :].reset_index(drop=True)
    spectra_matches = (preposs_data_raman.
                       query('id_file == @id_').
                       reset_index(drop=True).
                       merge(coord_matches, left_index=True, right_index=True).iloc[:, -15:])
    
    spectra_matches['distance'] = np.sqrt((spectra_matches['map_x_x'] - spectra_matches['map_x_y']) ** 2 + 
                                          (spectra_matches['map_y_x'] - spectra_matches['map_y_y']) ** 2)
    spectra_matches_list.append(spectra_matches)
    #coord_matches = coord_df.iloc[id_matches, :].reset_index(drop=True)
    
spectra_matches_all = pd.concat(spectra_matches_list)
spectra_matches_all = spectra_matches_all.query('distance<1').reset_index(drop=True)
spectra_matches_all#.query('id_file_y=="S1"')#.query('distance<1').to_csv('spectra_matches.csv')

,map_x_x,map_y_x,id_x,wn_saturated,no_signal,id_file_x,feat_id,map_x_y,map_y_y,size,area,perimeter,circularity,id_y,id_file_y,distance
0,21633.500000,-684.049988,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20240522_1,1450,0,S2/20240522,1,21633.68,-684.5956,5.794055,25.650240,19.410630,0.855280,S2/20240522/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20...,S2/20240522,0.574537
1,21621.400391,-656.900024,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20240522_2,0,1,S2/20240522,2,21621.84,-657.0768,7.697617,43.783180,28.313020,0.686169,S2/20240522/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20...,S2/20240522,0.473821
2,21584.650391,-627.150024,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20240522_3,1450,0,S2/20240522,3,21584.59,-627.3816,6.851129,35.918800,22.887980,0.861397,S2/20240522/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20...,S2/20240522,0.239320
3,21620.599609,-622.900024,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20240522_4,0,0,S2/20240522,4,21620.89,-623.4356,6.768998,35.859540,22.336000,0.903005,S2/20240522/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20...,S2/20240522,0.609236
4,21706.449219,-609.349976,MEC_02_119_DC_Sp_S2_GridSi_FOV1_20240522_5,0,0,S2/20240522,5,21706.68,-609.5938,6.852499,35.495530,23.103540,0.835436,S2/20240522/MEC_02_119_DC_Sp_S2_GridSi_FOV1_20...,S2/20240522,0.335724
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
961,17712.400391,-3408.000000,MEC_02_119_DC_Active_20240418_S6_ThinSi_202410...,0,0,S6/ThinSi,79,17712.09,-3407.7850,0.806646,0.448667,3.001583,0.625634,S6/ThinSi/DC_Active_20240418_S6_ThinSi_FOV2_20...,S6/ThinSi,0.377581
962,17606.900391,-3400.350098,MEC_02_119_DC_Active_20240418_S6_ThinSi_202410...,0,0,S6/ThinSi,80,17606.66,-3400.6630,1.200693,1.049713,4.334167,0.702030,S6/ThinSi/DC_Active_20240418_S6_ThinSi_FOV2_20...,S6/ThinSi,0.394583
963,17761.949219,-3396.600098,MEC_02_119_DC_Active_20240418_S6_ThinSi_202410...,0,0,S6/ThinSi,81,17762.05,-3396.8350,0.552013,0.211636,2.328753,0.490273,S6/ThinSi/DC_Active_20240418_S6_ThinSi_FOV2_20...,S6/ThinSi,0.255609
964,17750.900391,-3396.649902,MEC_02_119_DC_Active_20240418_S6_ThinSi_202410...,0,0,S6/ThinSi,82,17751.09,-3396.3820,1.202301,1.134367,5.070132,0.554385,S6/ThinSi/DC_Active_20240418_S6_ThinSi_FOV2_20...,S6/ThinSi,0.328212


In [23]:
preposs_data_raman_new = preposs_data_raman.merge(spectra_matches_all.loc[:,['id_x','size','area','perimeter','circularity','distance']], 
                                          how='left', left_on="id", right_on="id_x").reset_index(drop=True)
preposs_data_raman_new.drop('id_x', axis=1).to_csv('active_data_MEC_02_119_raman_specy_particle_info.csv')
